# APR from `patched_code.csv` (Colab)

Upload two files when prompted:

1. **`patched_code.csv`** (e.g. from `APR/PATCH GENERATION/`) — code + fault fields used for repair prompts.
2. **`final_dataset_v2.csv`** — use the same schema as `FED/HumanEval/HE_V3/final_dataset_v2.csv`. It defines **which error types exist per dataset**; the notebook picks rows **deterministically** (no random sampling): for every distinct `(dataset, error_types)` in that reference, it selects the reference row with the **smallest `task_id`** (numeric-aware), then uses that task in the patched file if present, otherwise the **first patched row** (by `task_id`) with the **same repair category**.

This gives a **static, reproducible** set of exemplars across all error labels that appear in the reference for each benchmark—useful for evaluating whether automated repair helps (e.g. showing persistent failure).

The notebook merges patched rows with **HumanEval**, **MBPP (sanitized)**, and **DS-1000** from Hugging Face for task text, runs a **Qwen** (or your chosen) instruct model with the same APR prompts as `PHASE_1_RUN.ipynb`, and downloads **`repair_results.csv`**.

**Runtime:** GPU recommended (e.g. T4).

In [ ]:
!pip install -q datasets transformers accelerate torch bitsandbytes

In [ ]:
import ast
import io
import json
import re
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    from google.colab import files
    _IN_COLAB = True
except ImportError:
    files = None
    _IN_COLAB = False

# --- config ---
MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
MAX_NEW_TOKENS = 512
REPAIR_TEMPERATURE = 0.7

In [ ]:
# Upload patched_code.csv + final_dataset_v2.csv (Colab).
# Local: set PATCHED_CSV_PATH and FINAL_DATASET_V2_PATH.

PATCHED_CSV_PATH: Optional[str] = None
FINAL_DATASET_V2_PATH: Optional[str] = None  # e.g. FED/HumanEval/HE_V3/final_dataset_v2.csv

if _IN_COLAB:
    print("1) Upload patched_code.csv")
    up1 = files.upload()
    if not up1:
        raise RuntimeError("patched_code.csv not uploaded.")
    n1 = next(iter(up1.keys()))
    df_patched = pd.read_csv(io.BytesIO(up1[n1]))
    print(f"   Loaded {len(df_patched)} rows from {n1!r}")
    print("2) Upload final_dataset_v2.csv (reference error-type universe)")
    up2 = files.upload()
    if not up2:
        raise RuntimeError("final_dataset_v2.csv not uploaded.")
    n2 = next(iter(up2.keys()))
    df_reference = pd.read_csv(io.BytesIO(up2[n2]))
    print(f"   Loaded {len(df_reference)} rows from {n2!r}")
elif PATCHED_CSV_PATH and FINAL_DATASET_V2_PATH:
    df_patched = pd.read_csv(PATCHED_CSV_PATH)
    df_reference = pd.read_csv(FINAL_DATASET_V2_PATH)
    print(f"Patched: {len(df_patched)} from {PATCHED_CSV_PATH!r}")
    print(f"Reference: {len(df_reference)} from {FINAL_DATASET_V2_PATH!r}")
else:
    raise RuntimeError(
        "Colab: upload both CSVs. Local: set PATCHED_CSV_PATH and FINAL_DATASET_V2_PATH."
    )

df_patched["task_id"] = df_patched["task_id"].astype(str)
df_reference["task_id"] = df_reference["task_id"].astype(str)

In [ ]:
# Benchmark tables (same sources as PHASE_1_RUN)

ds_he = load_dataset("openai/openai_humaneval")
df_humaneval = ds_he["test"].to_pandas()
df_humaneval["task_id"] = df_humaneval["task_id"].astype(str)

sanitized = load_dataset("google-research-datasets/mbpp", "sanitized")
mbpp_df = sanitized["train"].to_pandas()


def extract_signature(code: str):
    for line in code.splitlines():
        line = line.strip()
        if line.startswith("def "):
            return line
    return ""


mbpp_df["function_signature"] = mbpp_df["code"].apply(extract_signature)
mbpp_df["task_id"] = mbpp_df["task_id"].astype(str)

ds1k = load_dataset("xlangai/DS-1000")
df_ds1k = ds1k["test"].to_pandas()
df_ds1k["prompt_2"] = df_ds1k["prompt"].str.split("A:", n=1).str[0].str.strip()
df_ds1k["task_id"] = [f"DS{str(i).zfill(4)}" for i in range(len(df_ds1k))]

print("HumanEval rows:", len(df_humaneval), "MBPP:", len(mbpp_df), "DS1000:", len(df_ds1k))

In [ ]:
def merge_patched_with_prompts(df_patch: pd.DataFrame) -> pd.DataFrame:
    """Inner-join patched rows with benchmark prompts."""
    out = []
    for ds in df_patch["dataset"].unique():
        sub = df_patch[df_patch["dataset"] == ds].copy()
        if ds == "HumanEval":
            m = sub.merge(df_humaneval, on="task_id", how="inner", suffixes=("", "_he"))
            m["prompt_text"] = m["prompt"].astype(str)
        elif ds == "MBPP":
            m = sub.merge(mbpp_df, on="task_id", how="inner", suffixes=("", "_mbpp"))
            m["prompt_text"] = (
                m["prompt"].astype(str) + "\n\nImplement:\n" + m["function_signature"].astype(str)
            )
        elif ds == "DS1000":
            m = sub.merge(df_ds1k, on="task_id", how="inner", suffixes=("", "_ds"))
            m["prompt_text"] = m["prompt_2"].fillna(m["prompt"]).astype(str)
        else:
            continue
        out.append(m)
    if not out:
        return pd.DataFrame()
    return pd.concat(out, ignore_index=True)


df_merged = merge_patched_with_prompts(df_patched)
print("Merged rows (with prompt):", len(df_merged))
if len(df_merged) == 0:
    raise RuntimeError("No rows merged — check task_id/dataset alignment with HF splits.")

In [ ]:
def _normalize_error_type(s: str) -> str:
    t = s.split(":")[-1].strip() if ":" in s else s.strip()
    return t.replace(" ", "")


def get_repair_category(error_types: str, fault_information: dict) -> str:
    if not error_types or not str(error_types).strip():
        return "skip"
    parts = [p.strip() for p in str(error_types).split(",")]
    for p in parts:
        t = _normalize_error_type(p)
        if t in ("SyntaxError", "IndentationError"):
            return "syntax"
        if t in ("AttributeError", "attribute_error"):
            return "attribute"
        if t == "TypeError":
            return "type"
        if t in ("NameError", "name_error"):
            return "name"
        if t == "KeyError":
            return "key"
        if t in ("AssertionError", "WrongAnswer"):
            return "assertion"
        if t == "TimeoutError":
            return "timeout"
    return "other"


def _parse_line_no(v) -> Optional[int]:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    s = str(v).strip()
    if not s:
        return None
    try:
        return int(float(s))
    except (ValueError, TypeError):
        return None


def fault_information_from_csv_row(row: pd.Series) -> Dict[str, Any]:
    """Map patched_code.csv row → fault_information dict for APR builders."""
    dataset = row.get("dataset")
    task_id = str(row.get("task_id", ""))
    status = row.get("status")

    ast_info = None
    raw_ast = row.get("ast_info")
    if pd.notna(raw_ast) and str(raw_ast).strip():
        try:
            j = json.loads(raw_ast) if isinstance(raw_ast, str) else raw_ast
            if isinstance(j, dict) and "value" in j:
                ln = int(j["value"]) if j["value"] is not None else None
                ast_info = {
                    "ast_errors": [
                        {
                            "type": j.get("type", "AST_Error"),
                            "message": j.get("message", ""),
                            "start_line": ln,
                            "end_line": ln,
                        }
                    ]
                }
        except (json.JSONDecodeError, TypeError, ValueError, KeyError):
            ast_info = None

    dynamic_info = None
    raw_dyn = row.get("dynamic_info")
    if pd.notna(raw_dyn) and str(raw_dyn).strip():
        try:
            d = json.loads(raw_dyn) if isinstance(raw_dyn, str) else raw_dyn
            if isinstance(d, dict):
                d = dict(d)
                if d.get("error_type") or d.get("error_message"):
                    d["status"] = "failed"
                if d.get("line_number") in (None, "") and "line_no" in d:
                    d["line_number"] = _parse_line_no(d.get("line_no"))
                dynamic_info = d
        except (json.JSONDecodeError, TypeError):
            dynamic_info = None

    lib_info = None
    raw_lib = row.get("lib_info")
    if pd.notna(raw_lib) and str(raw_lib).strip():
        try:
            lib_info = ast.literal_eval(str(raw_lib))
        except (ValueError, SyntaxError):
            lib_info = None

    return {
        "dataset": dataset,
        "task_id": task_id,
        "status": status,
        "ast_info": ast_info,
        "lib_info": lib_info,
        "dynamic_info": dynamic_info,
    }


def _parse_dynamic_info(fault_information: dict) -> dict:
    d = fault_information.get("dynamic_info")
    if d is None:
        return {}
    if isinstance(d, dict):
        return d
    try:
        return json.loads(d) if isinstance(d, str) else {}
    except Exception:
        return {}


def extract_error_info_for_prompt(fault_information: dict) -> dict:
    dyn = _parse_dynamic_info(fault_information or {})
    return {
        "error_type": dyn.get("error_type", ""),
        "error_message": dyn.get("error_message", ""),
        "line_number": dyn.get("line_number", ""),
    }


def extract_failing_test_cases_for_prompt(fault_information: dict, max_cases: int = 10) -> list:
    dyn = _parse_dynamic_info(fault_information or {})
    raw = dyn.get("test_case", "")
    if not raw:
        return []
    if isinstance(raw, str) and raw.strip().startswith("["):
        try:
            raw = json.loads(raw)
        except Exception:
            return []
    if not isinstance(raw, list):
        return []
    out = []
    for item in raw[:max_cases]:
        if not isinstance(item, (list, tuple)) or len(item) < 3:
            continue
        input_str, expected_str, actual_str = str(item[0]), str(item[1]), str(item[2])
        if expected_str == actual_str:
            continue
        out.append(
            f"For this INPUT {input_str} we get output {actual_str} but we need this {expected_str}"
        )
    return out


def build_prompt_syntax(
    patched_code: str, fault_information: dict, original_question: str, suggestions=None
) -> list:
    ast_info = fault_information.get("ast_info") or {}
    dynamic_info = fault_information.get("dynamic_info") or {}
    error_message = ""
    line_info = ""

    if ast_info and ast_info.get("ast_errors"):
        err = ast_info["ast_errors"][0]
        error_message = err.get("message", "Syntax or structural error")
        line_info = f"Line(s) {err.get('start_line', '?')}-{err.get('end_line', err.get('start_line', '?'))}"
    elif dynamic_info and dynamic_info.get("status") == "failed":
        error_message = dynamic_info.get("error_message", "Syntax or runtime error")
        line_info = (
            f"Line {dynamic_info.get('line_number', '?')}"
            if dynamic_info.get("line_number")
            else ""
        )

    system_message = (
        "You are an expert Python developer. Your ONLY task is to fix syntax or indentation errors.\n"
        "You must NOT change program logic, add new features, or refactor.\n"
        "Fix ONLY the reported line(s). Remove any ERROR markers from the output.\n"
        "Return ONLY a single Python code block wrapped in ```python and ```. No explanations."
    )
    user_content = (
        f"Original task or context:\n{original_question[:1500]}\n\n"
        f"Error: {error_message}\n{line_info}\n\n"
        "Buggy code (with optional error markers):\n"
        f"```\n{patched_code}\n```\n\n"
        "Fix the syntax/indentation only. Output the complete corrected code in one ```python block."
    )
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_content},
    ]


def build_prompt_attribute(
    patched_code: str, fault_information: dict, original_question: str, suggestions=None
) -> list:
    err_info = extract_error_info_for_prompt(fault_information)
    err_msg = err_info.get("error_message") or "Attribute error"
    system = (
        "You are a Python expert. Fix ONLY the attribute/API error. "
        "Replace wrong or missing attribute with the correct one. "
        "Return ONLY one ```python code block. No explanations."
    )
    user = (
        f"Task:\n{original_question[:1200]}\n\nError: {err_msg}\n\nBuggy code:\n```\n{patched_code}\n```\n\n"
        "Fix the attribute error. Output full corrected code in one ```python block. Remove any ERROR markers."
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_prompt_type(
    patched_code: str, fault_information: dict, original_question: str, suggestions=None
) -> list:
    err_info = extract_error_info_for_prompt(fault_information)
    err_msg = err_info.get("error_message") or "Type error"
    system = (
        "You are a Python expert. Fix ONLY the type/signature error (e.g. wrong keyword, wrong argument type). "
        "Return ONLY one ```python code block. No explanations."
    )
    user = (
        f"Task:\n{original_question[:1200]}\n\nError: {err_msg}\n\nBuggy code:\n```\n{patched_code}\n```\n\n"
        "Fix the type or call signature. Output full corrected code in one ```python block. Remove any ERROR markers."
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_prompt_name(
    patched_code: str, fault_information: dict, original_question: str, suggestions=None
) -> list:
    err_info = extract_error_info_for_prompt(fault_information)
    err_msg = err_info.get("error_message") or "Name not defined"
    system = (
        "You are a Python expert. Fix ONLY the NameError: add the missing import or define the missing variable. "
        "Return ONLY one ```python code block. No explanations."
    )
    user = (
        f"Task:\n{original_question[:1200]}\n\nError: {err_msg}\n\nBuggy code:\n```\n{patched_code}\n```\n\n"
        "Add missing import or definition. Output full corrected code in one ```python block. Remove any ERROR markers."
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_prompt_key(
    patched_code: str, fault_information: dict, original_question: str, suggestions=None
) -> list:
    err_info = extract_error_info_for_prompt(fault_information)
    err_msg = err_info.get("error_message") or "Key error"
    system = (
        "You are a Python expert. Fix ONLY the KeyError: use the correct key or handle missing key. "
        "Return ONLY one ```python code block. No explanations."
    )
    user = (
        f"Task:\n{original_question[:1200]}\n\nError: {err_msg}\n\nBuggy code:\n```\n{patched_code}\n```\n\n"
        "Fix the key access. Output full corrected code in one ```python block. Remove any ERROR markers."
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_prompt_assertion(
    patched_code: str, fault_information: dict, original_question: str, suggestions=None
) -> list:
    failing = extract_failing_test_cases_for_prompt(fault_information)
    test_str = "\n".join(failing) if failing else "(no failing test details)"
    system = (
        "You are a Python expert. Fix the LOGIC so the tests pass. Do NOT change function signature or add helpers. "
        "Modify minimum lines. Return ONLY one ```python code block. No explanations."
    )
    user = (
        f"Task:\n{original_question[:1500]}\n\nBuggy code (fails tests):\n```\n{patched_code}\n```\n\n"
        f"Failing tests (change logic accordingly):\n{test_str}\n\n"
        "Fix the logic only. Output full corrected code in one ```python block. Remove any ERROR markers."
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_prompt_timeout(
    patched_code: str, fault_information: dict, original_question: str, suggestions=None
) -> list:
    err_info = extract_error_info_for_prompt(fault_information)
    err_msg = err_info.get("error_message") or "Timeout"
    system = (
        "You are a Python expert. Fix the timeout: ensure loops terminate or reduce work (e.g. avoid brute force). "
        "Return ONLY one ```python code block. No explanations."
    )
    user = (
        f"Task:\n{original_question[:1200]}\n\nError: {err_msg}\n\nBuggy code:\n```\n{patched_code}\n```\n\n"
        "Fix so it completes in time. Output full corrected code in one ```python block. Remove any ERROR markers."
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_prompt_other(
    patched_code: str, fault_information: dict, original_question: str, suggestions=None
) -> list:
    err_info = extract_error_info_for_prompt(fault_information)
    err_msg = err_info.get("error_message")
    if not err_msg:
        ast_info = fault_information.get("ast_info") or {}
        if ast_info.get("ast_errors"):
            err_msg = ast_info["ast_errors"][0].get("message", "Error")
    if not err_msg:
        err_msg = "Runtime or structural error"
    system = (
        "You are a Python expert. Fix the error in the marked region or minimal lines. "
        "Return ONLY one ```python code block. No explanations."
    )
    user = (
        f"Task:\n{original_question[:1200]}\n\nError: {err_msg}\n\nBuggy code:\n```\n{patched_code}\n```\n\n"
        "Fix the error. Output full corrected code in one ```python block. Remove any ERROR markers."
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_apr_prompt(
    patched_code: str,
    fault_information: dict,
    original_question: str,
    error_types: str,
    suggestions=None,
) -> list:
    category = get_repair_category(error_types or "", fault_information or {})
    if category == "skip":
        return []
    builders = {
        "syntax": build_prompt_syntax,
        "attribute": build_prompt_attribute,
        "type": build_prompt_type,
        "name": build_prompt_name,
        "key": build_prompt_key,
        "assertion": build_prompt_assertion,
        "timeout": build_prompt_timeout,
        "other": build_prompt_other,
    }
    fn = builders.get(category, build_prompt_other)
    return fn(patched_code, fault_information or {}, original_question, suggestions)


def extract_python_code_humaneval(text: str) -> str:
    pattern = r"```(?:python)?\n?(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()


def extract_python_code_mbpp(text: str) -> str:
    return extract_python_code_humaneval(text)

In [ ]:
def effective_error_types(row: pd.Series, fault_information: dict) -> str:
    """Avoid skip when CSV error_types is empty but dynamic_info has error_type."""
    et = row.get("error_types")
    if pd.notna(et) and str(et).strip():
        return str(et)
    dyn = fault_information.get("dynamic_info") or {}
    t = dyn.get("error_type")
    return str(t) if t else ""


def normalize_dataset_label(ds) -> str:
    if pd.isna(ds):
        return ""
    k = str(ds).strip().lower()
    if k == "humaneval":
        return "HumanEval"
    if k == "mbpp":
        return "MBPP"
    if k in ("ds1000", "ds-1000"):
        return "DS1000"
    return str(ds).strip()


def _task_sort_key(tid: str) -> Tuple:
    """Stable ordering: HumanEval/9 before HumanEval/10; DS0002 before DS0010; MBPP numeric."""
    s = str(tid)
    if "/" in s:
        pref, _, rest = s.partition("/")
        try:
            return (0, pref, int(rest))
        except ValueError:
            return (0, pref, rest)
    if s.upper().startswith("DS"):
        tail = s[2:]
        try:
            return (1, int(tail))
        except ValueError:
            return (1, tail)
    try:
        return (2, int(s))
    except ValueError:
        return (2, s)


def _min_task_id(ids) -> str:
    return min((str(x) for x in ids), key=_task_sort_key)


def category_from_reference_error_types(error_types: str) -> str:
    """Repair category for short reference labels (NameError, SyntaxError, ...)."""
    return get_repair_category(str(error_types) if error_types is not None else "", {})


def select_static_from_reference(df_merged: pd.DataFrame, df_ref: pd.DataFrame) -> pd.DataFrame:
    """
    No sampling. For each distinct (dataset, error_types) in df_ref with non-empty error_types,
    pick the reference task_id = min task_id for that pair, then:
      - use that row from df_merged if present;
      - else first df_merged row (by task_id) with same dataset and same repair category.
    """
    ref = df_ref.copy()
    ref["dataset_n"] = ref["dataset"].map(normalize_dataset_label)
    ref["task_id"] = ref["task_id"].astype(str)
    ref = ref[ref["error_types"].notna() & ref["error_types"].astype(str).str.strip().ne("")]

    pairs = ref[["dataset_n", "error_types"]].drop_duplicates()
    pairs = pairs.sort_values(["dataset_n", "error_types"])

    picked_indices: List[int] = []
    log = []

    for _, pr in pairs.iterrows():
        dsn = pr["dataset_n"]
        et = str(pr["error_types"])
        sub = ref[(ref["dataset_n"] == dsn) & (ref["error_types"].astype(str) == et)]
        tid_ref = _min_task_id(sub["task_id"].unique())
        cat = category_from_reference_error_types(et)

        hit = df_merged[(df_merged["dataset"] == dsn) & (df_merged["task_id"] == tid_ref)]
        if len(hit) >= 1:
            idx = hit.index[0]
            how = "exact_task_from_reference"
        else:
            pool = df_merged[df_merged["dataset"] == dsn]
            if pool.empty:
                log.append((dsn, et, tid_ref, cat, None, "no_merged_rows_for_dataset"))
                continue
            candidates = []
            for idx, row in pool.iterrows():
                fi = fault_information_from_csv_row(row)
                ec = effective_error_types(row, fi)
                rc = get_repair_category(ec, fi)
                if rc == cat and rc != "skip":
                    candidates.append((row["task_id"], idx))
            if not candidates:
                log.append((dsn, et, tid_ref, cat, None, "no_category_match_in_patched"))
                continue
            candidates.sort(key=lambda x: _task_sort_key(x[0]))
            idx = candidates[0][1]
            how = "fallback_same_repair_category"

        picked_indices.append(idx)
        log.append((dsn, et, tid_ref, cat, df_merged.loc[idx, "task_id"], how))

    if not picked_indices:
        raise RuntimeError("No rows selected — check reference vs patched overlap.")

    df_run = df_merged.loc[sorted(set(picked_indices))].copy()

    cats = []
    for _, row in df_run.iterrows():
        fi = fault_information_from_csv_row(row)
        et = effective_error_types(row, fi)
        cats.append(get_repair_category(et, fi))
    df_run["_repair_category"] = cats

    print("Reference-derived (dataset, error_types) pairs:", len(pairs))
    print("Rows selected for repair (deterministic):", len(df_run))
    print("\nSelection log (dataset, ref_error_types, ref_min_task_id, category, chosen_task_id, how):")
    for row in log:
        print(" ", row)
    print("\nCounts by dataset × _repair_category in run set:")
    print(df_run.groupby(["dataset", "_repair_category"]).size())

    return df_run.reset_index(drop=True)


df_run = select_static_from_reference(df_merged, df_reference)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

_device = next(model.parameters()).device
print(f"Loaded {MODEL_ID} on {_device}")

In [ ]:
def generate_code_repair(formatted_messages: list, do_sample: bool = True, temperature: float = None) -> str:
    if temperature is None:
        temperature = REPAIR_TEMPERATURE
    inputs = tokenizer.apply_chat_template(
        formatted_messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(_device)
    gen_kw = {"max_new_tokens": MAX_NEW_TOKENS, "pad_token_id": tokenizer.eos_token_id}
    if do_sample:
        gen_kw["do_sample"] = True
        gen_kw["temperature"] = temperature
    else:
        gen_kw["do_sample"] = False
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kw)
    gen_ids = outputs[0][len(inputs["input_ids"][0]) :]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)


def repair_one_row(row: pd.Series) -> Dict[str, Any]:
    patch = str(row.get("patched_code", "") or "")
    generated_code = str(row.get("generated_code", "") or "")
    if not patch:
        patch = generated_code
    fault_information = fault_information_from_csv_row(row)
    et = effective_error_types(row, fault_information)
    original_question = str(row.get("prompt_text", ""))[:2000]
    messages = build_apr_prompt(patch, fault_information, original_question, et, None)
    if not messages:
        return {
            "repaired_code": generated_code,
            "raw_response": "",
            "repair_category": row.get("_repair_category"),
            "skipped": True,
        }
    raw = generate_code_repair(messages, do_sample=True, temperature=REPAIR_TEMPERATURE)
    ds = row.get("dataset")
    extract_fn = extract_python_code_mbpp if ds == "MBPP" else extract_python_code_humaneval
    fixed = extract_fn(raw)
    out_code = fixed.strip() if fixed and fixed.strip() else generated_code
    return {
        "repaired_code": out_code,
        "raw_response": raw,
        "repair_category": get_repair_category(et, fault_information),
        "skipped": False,
    }

In [ ]:
results = []
for i, row in df_run.iterrows():
    print(f"[{i+1}/{len(df_run)}] {row['dataset']} {row['task_id']} ...")
    r = repair_one_row(row)
    results.append(
        {
            "dataset": row["dataset"],
            "task_id": row["task_id"],
            "error_types": row.get("error_types"),
            "error_sources": row.get("error_sources"),
            "repair_category": r["repair_category"],
            "skipped": r["skipped"],
            "patched_code": row.get("patched_code"),
            "repaired_code": r["repaired_code"],
            "raw_response": r["raw_response"],
        }
    )

results_df = pd.DataFrame(results)
out_name = "repair_results.csv"
results_df.to_csv(out_name, index=False)
print("Wrote", out_name, "—", len(results_df), "rows")

if _IN_COLAB:
    files.download(out_name)